# Analyze Eval Reports

In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import fsspec
import pandas as pd

from llm_agent.evaluation.base_eval import EvalReport

In [ ]:
from os import listdir
from os.path import isfile, join

eval_report_dir = "../evals/outputs/"
eval_report_paths = [f for f in listdir(eval_report_dir) if isfile(join(eval_report_dir, f))]
eval_report_paths

In [ ]:
eval_reports = []
for path in eval_report_paths:
    fs, path = fsspec.url_to_fs(f"../evals/outputs/{path}")
    with fs.open(path) as f:
        eval_report = EvalReport.model_validate_json(f.read())
        eval_reports.append(eval_report)

In [ ]:
evals_df = pd.DataFrame.from_records([
    dict(
        experiment_name=eval_report.experiment_config.name, 
        **eval_report.experiment_config.params, 
        **eval_report.aggregated_metrics,
    )
    for eval_report in eval_reports
])
evals_df

In [ ]:
pd.DataFrame.from_records([
    dict(**report_haiku.experiment_config.params, **report_haiku.aggregated_metrics),
    dict(**report_sonnet.experiment_config.params, **report_sonnet.aggregated_metrics),
])